# Calorie model – Sprint 1 starter
Covers: (1) load and inspect the data, (2) MET-formula baseline, (3) Linear Regression and Random Forest.
Edit the paths in the next cell, then run top to bottom. Write your own observations in the *Notes* cells – those become your report.

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ---- CONFIG: point these at your downloaded Kaggle files ----
# Some versions of the dataset ship as two files (exercise.csv + calories.csv, joined on User_ID),
# others as one file. If you have ONE file, set EXERCISE_CSV = None.
CALORIES_CSV = "../data/raw/fmendes-DAT263x-demos/calories.csv"
EXERCISE_CSV = "../data/raw/fmendes-DAT263x-demos/exercise.csv"
RANDOM_STATE = 42

## 1. Load and inspect

In [2]:
df = pd.read_csv(CALORIES_CSV)
if EXERCISE_CSV:
    df = pd.read_csv(EXERCISE_CSV).merge(df, on="User_ID", how="inner")
print(df.shape)
df.head()

(15000, 9)


,User_ID,Gender,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Calories
0,14733363,male,68,190.0,94.0,29.0,105.0,40.8,231.0
1,14861698,female,20,166.0,60.0,14.0,94.0,40.3,66.0
2,11179863,male,69,179.0,79.0,5.0,88.0,38.7,26.0
3,16180408,female,34,179.0,71.0,13.0,100.0,40.5,71.0
4,17771927,female,27,154.0,58.0,10.0,81.0,39.8,35.0


In [3]:
print(df.dtypes, "\n")
print("Missing values:\n", df.isna().sum(), "\n")
print("Duplicate rows:", df.duplicated().sum())
df.describe().T

User_ID         int64
Gender            str
Age             int64
Height        float64
Weight        float64
Duration      float64
Heart_Rate    float64
Body_Temp     float64
Calories      float64
dtype: object 

Missing values:
 User_ID       0
Gender        0
Age           0
Height        0
Weight        0
Duration      0
Heart_Rate    0
Body_Temp     0
Calories      0
dtype: int64 

Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
User_ID,15000.0,1.497736e+07,2.872851e+06,10001159.0,12474190.75,14997285.0,17449278.75,19999647.0
Age,15000.0,4.278980e+01,1.698026e+01,20.0,28.00,39.0,56.00,79.0
Height,15000.0,1.744651e+02,1.425811e+01,123.0,164.00,175.0,185.00,222.0
Weight,15000.0,7.496687e+01,1.503566e+01,36.0,63.00,74.0,87.00,132.0
Duration,15000.0,1.553060e+01,8.319203e+00,1.0,8.00,16.0,23.00,30.0
Heart_Rate,15000.0,9.551853e+01,9.583328e+00,67.0,88.00,96.0,103.00,128.0
Body_Temp,15000.0,4.002545e+01,7.792299e-01,37.1,39.60,40.2,40.60,41.5
Calories,15000.0,8.953953e+01,6.245698e+01,1.0,35.00,79.0,138.00,314.0


In [4]:
# Outlier check (IQR rule) – report, don't silently delete
num = df.select_dtypes("number").drop(columns=["User_ID"], errors="ignore")
q1, q3 = num.quantile(.25), num.quantile(.75); iqr = q3 - q1
outliers = ((num < q1 - 1.5*iqr) | (num > q3 + 1.5*iqr)).sum()
print("Rows outside 1.5*IQR per column:\n", outliers)
# Physically impossible values
bad = df[(df["Duration"] <= 0) | (df["Weight"] <= 0) | (df["Age"] <= 0)]
print("\nImpossible rows:", len(bad))

Rows outside 1.5*IQR per column:
 Age             0
Height         14
Weight          6
Duration        0
Heart_Rate      1
Body_Temp     369
Calories        4
dtype: int64

Impossible rows: 0


**Notes (write your own):** what did you find? Which columns are suspicious? What will you do about outliers and why?

## 2. MET-formula baseline
`kcal = MET × weight (kg) × duration (hours)`.
**Assumption:** the dataset doesn't say which activity was done, so one constant MET is used. `MET = 5.0` is a placeholder (moderate effort) – change it and justify your value with a source (e.g. the Compendium of Physical Activities).

In [ ]:
MET = 5.0
FEATURES_FULL = ["Gender", "Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]
FEATURES_APP  = ["Gender", "Age", "Height", "Weight", "Duration"]   # what the app can collect without sensors
TARGET = "Calories"

d = df.copy()
d["Gender"] = d["Gender"].astype(str).str.lower().map({"male": 1, "female": 0})
assert d["Gender"].notna().all(), "Unexpected Gender values"

train, test = train_test_split(d, test_size=0.2, random_state=RANDOM_STATE)

def scores(y, p):
    return {"MSE": mean_squared_error(y, p), "RMSE": mean_squared_error(y, p) ** .5,
            "MAE": mean_absolute_error(y, p), "R2": r2_score(y, p)}

results = {}
results["MET baseline"] = scores(test[TARGET], MET * test["Weight"] * test["Duration"] / 60)
pd.DataFrame(results).T

*Check:* Duration is assumed to be in **minutes** (hence `/60`). Confirm that from the dataset description.

## 3. Linear Regression and Random Forest
Each model is trained on two feature sets: **full** (includes heart rate and body temperature) and **app-only** (what your app can actually collect). The gap between them shows how much the sensor features matter – this feeds directly into your Section 3.3 argument.

In [ ]:
models = {"Linear Regression": LinearRegression(),
          "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)}
fitted = {}
for fs_name, feats in [("full", FEATURES_FULL), ("app-only", FEATURES_APP)]:
    for m_name, m in models.items():
        m.fit(train[feats], train[TARGET])
        pred = m.predict(test[feats])
        results[f"{m_name} ({fs_name})"] = scores(test[TARGET], pred)
        fitted[(m_name, fs_name)] = pred
res = pd.DataFrame(results).T.round(3)
res

In [ ]:
# Predicted vs actual for the best full-feature model
p = fitted[("Random Forest", "full")]
plt.scatter(test[TARGET], p, s=4, alpha=.4); lim = [0, test[TARGET].max()]
plt.plot(lim, lim, "r--"); plt.xlabel("Actual kcal"); plt.ylabel("Predicted kcal")
plt.title("Random Forest (full features)"); plt.show()

In [ ]:
# First bias check: error by gender (extend to age groups later)
t = test.copy(); t["abs_err"] = (t[TARGET] - fitted[("Random Forest", "app-only")]).abs()
t.groupby("Gender")["abs_err"].agg(["mean", "count"]).rename(index={0: "female", 1: "male"})

**Notes (write your own):**
- How much does each model beat the MET baseline?
- If R² is ~0.99 with full features, what does that say about this dataset?
- How much accuracy is lost without heart rate and body temperature?
- Is the error noticeably different between groups?